In [41]:
from pathlib import Path
import asyncio
import base64
from typing import Optional

import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, display

workspace_root = Path.cwd().resolve()
if workspace_root.name.lower() == "notebooks":
    workspace_root = workspace_root.parent

homes_file = workspace_root / "notebooks" / "Daily_Usage_Homes.xlsx"
wind_file = workspace_root / "notebooks" / "Windturbine_Daily_Produced.xlsx"

homes_df = pd.read_excel(homes_file)
wind_df = pd.read_excel(wind_file)

homes_df.columns = [str(column).strip() for column in homes_df.columns]
wind_df.columns = [str(column).strip() for column in wind_df.columns]


def pick_column(df: pd.DataFrame, candidates: list[str], label: str) -> str:
    for name in candidates:
        if name in df.columns:
            return name
    raise KeyError(f"Missing {label} column. Found columns: {list(df.columns)}")


home_date_col = pick_column(homes_df, ["date", "Date"], "homes date")
wind_date_col = pick_column(wind_df, ["date", "Date"], "wind date")

home_value_col = pick_column(
    homes_df,
    ["homes_shared_energy_usage_kwh_day", "homes_shared_energy_usage_kWh", "usage_kWh"],
    "homes usage",
)
wind_value_col = pick_column(
    wind_df,
    ["Windturbine_Produced_kwh_day", "Windturbine_Produced", "Windturbine_Produced_Day"],
    "wind production",
)

homes_source = homes_df[[home_date_col, home_value_col]].rename(
    columns={home_date_col: "date", home_value_col: "homes_shared_energy_usage_kwh_day"}
)
wind_source = wind_df[[wind_date_col, wind_value_col]].rename(
    columns={wind_date_col: "date", wind_value_col: "Windturbine_Produced_kwh_day"}
)

homes_source["date"] = pd.to_datetime(homes_source["date"], errors="coerce").dt.strftime("%Y-%m-%d")
wind_source["date"] = pd.to_datetime(wind_source["date"], errors="coerce").dt.strftime("%Y-%m-%d")

daily_df = (
    wind_source.merge(homes_source, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)

if daily_df.empty:
    raise RuntimeError("No overlapping daily rows between wind and homes files.")

BATTERY_CAPACITY_KWH = 2_500_000.0
SECONDS_PER_DAY = 1.0
battery_storage_kwh = 25_000.0
pause_event = asyncio.Event()
pause_event.set()


def image_data_uri(file_name: str) -> str:
    image_path = workspace_root / file_name
    if not image_path.exists():
        return ""
    encoded = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"


battery_image = image_data_uri("image-1.png")
wind_image = image_data_uri("image-3.png")
homes_image = image_data_uri("image-4.png")

pause_toggle = widgets.ToggleButton(
    value=False,
    description="Pause",
    tooltip="Pause or resume simulation",
    icon="pause",
)
status_message = widgets.HTML()
status_visual = widgets.HTML()
status_panel = widgets.VBox([status_message, status_visual])
display(pause_toggle, status_panel)


def _on_pause_change(change: dict) -> None:
    if change["name"] != "value":
        return

    if change["new"]:
        pause_toggle.description = "Resume"
        pause_toggle.icon = "play"
        pause_event.clear()
    else:
        pause_toggle.description = "Pause"
        pause_toggle.icon = "pause"
        pause_event.set()


pause_toggle.observe(_on_pause_change, names="value")


def build_visual_html(daily_line: dict, paused: bool = False) -> str:
    pause_notice = ""
    if paused:
        pause_notice = (
            "<div style='margin-bottom:8px;color:#B54708;font-weight:600;'>"
            "Simulation paused. Press Resume to continue."
            "</div>"
        )

    battery_pct = max(0, min(100, int(daily_line["Battery_Percentage"])))
    battery_fill_width = int(round(72 * (battery_pct / 100)))
    wind_kwh_day = float(daily_line['Windturbine_Produced_kwh_day'])
    homes_kwh_day = float(daily_line['homes_shared_energy_usage_kwh_day'])
    if homes_kwh_day <= 0:
        direct_wind_share_pct = 0.0
    else:
        direct_wind_to_homes = min(max(wind_kwh_day, 0.0), homes_kwh_day)
        direct_wind_share_pct = min(100.0, (direct_wind_to_homes / homes_kwh_day) * 100.0)
    top_arrow_color = "#22c55e" if wind_kwh_day > homes_kwh_day else "#111827"
    arrow_size = 28
    left_arrow_color = "#22c5bd" if direct_wind_share_pct > 0 else "#111827"
    diagonal_arrow_color = "#22c55e" if homes_kwh_day > wind_kwh_day else "#111827"
    energy_lost_day_kwh = float(daily_line['Energy_Lost_Day_kwh'])
    energy_lost_total_kwh = float(daily_line.get('Energy_Lost_Total_kwh', energy_lost_day_kwh))
    day_loss_display = f"{energy_lost_day_kwh:.2f} kWh"
    total_loss_display = f"{energy_lost_total_kwh:.2f} kWh"
    excess_loss_label = (
        "<span style='font-size:10px;font-weight:600;color:#6b7280;'>Energy Loss</span><br>"
        f"<span style='display:inline-block;min-width:130px;font-variant-numeric:tabular-nums;'>Day: {day_loss_display}</span><br>"
        f"<span style='display:inline-block;min-width:130px;font-variant-numeric:tabular-nums;'>Total: {total_loss_display}</span>"
    )

    return f"""
<div style='margin-top:10px;padding:12px;border:1px solid #d1d5db;border-radius:10px;max-width:900px;position:relative;'>
  {pause_notice}
  <div style='position:absolute;top:28px;left:12px;font-size:12px;font-weight:600;color:#374151;'>{daily_line['date']}</div>
  <div style='position:absolute;top:28px;right:-20px;width:170px;font-size:12px;font-weight:700;color:#c52722;line-height:1.2;text-align:left;'>{excess_loss_label}</div>
  <div style='display:flex;justify-content:center;align-items:center;margin-bottom:10px;'>
    <div style='text-align:center;'>
      <div style='position:relative;width:160px;height:80px;margin:0 auto;'>
        <div style='position:absolute;left:45px;top:24px;width:72px;height:30px;z-index:1;overflow:hidden;clip-path:polygon(0 0, 88% 0, 88% 18%, 100% 18%, 100% 82%, 88% 82%, 88% 100%, 0 100%);background:#e5e7eb;'>
          <div style='height:100%;width:{battery_fill_width}px;background:#22c55e;transition:width 0.25s linear;'></div>
        </div>
        <img src='{battery_image}' style='position:absolute;inset:0;width:160px;height:80px;object-fit:contain;z-index:2;'/>
      </div>
      <div style='font-weight:700;'>Battery</div>
      <div>{battery_pct}%</div>
    </div>
  </div>
  <div style='display:grid;grid-template-columns:1fr 80px 1fr;align-items:center;gap:8px;'>
    <div style='text-align:center;'>
      <img src='{wind_image}' style='height:90px;object-fit:contain;'/>
      <div style='font-weight:700;'>Windturbine</div>
      <div>Produced: {daily_line['Windturbine_Produced_kwh_day']:.2f} kWh/day</div>
    </div>
    <div style='display:flex;flex-direction:column;align-items:center;justify-content:center;gap:4px;position:relative;'>
      <div style='position:absolute;top:-78px;right:-66px;font-size:{arrow_size}px;line-height:1;font-weight:800;color:{diagonal_arrow_color};'>↘</div>
      <div style='font-size:{arrow_size}px;line-height:1;font-weight:900;color:{top_arrow_color};-webkit-text-stroke:1px currentColor;transform:translateY(-35px);'>↑</div>
      <div style='display:flex;align-items:center;gap:4px;'>
        <div style='font-size:{arrow_size}px;line-height:1;font-weight:900;color:{left_arrow_color};-webkit-text-stroke:1px currentColor;width:18px;text-align:center;transform:translateX(-30px);'>→</div>
        <div style='width:64px;height:10px;border:1px solid #9ca3af;border-radius:0;background:transparent;overflow:hidden;'>
          <div style='width:{direct_wind_share_pct}%;height:100%;background:#22c5bd;'></div>
        </div>
        <div style='font-size:{arrow_size}px;line-height:1;font-weight:900;-webkit-text-stroke:1px currentColor;transform:translateX(30px);'>→</div>
      </div>
    </div>
    <div style='text-align:center;'>
      <img src='{homes_image}' style='height:90px;object-fit:contain;'/>
      <div style='font-weight:700;'>Homes</div>
      <div>Usage: {daily_line['homes_shared_energy_usage_kwh_day']:.2f} kWh/day</div>
    </div>
  </div>
</div>
"""


def show_daily_output(
    daily_line: Optional[dict] = None,
    *,
    paused: bool = False,
    message: Optional[str] = None,
) -> None:
    if message:
        status_message.value = f"<pre>{message}</pre>"
        status_visual.value = ""
        return

    if daily_line is None:
        status_message.value = "<pre>Waiting for simulation data...</pre>"
        status_visual.value = ""
        return

    status_message.value = (
        "<pre>"
        f"date: {daily_line['date']}\n"
        f"Windturbine_Produced_kwh_day: {daily_line['Windturbine_Produced_kwh_day']:.2f}\n"
        f"homes_shared_energy_usage_kwh_day: {daily_line['homes_shared_energy_usage_kwh_day']:.2f}\n"
        f"Battery_Percentage: {daily_line['Battery_Percentage']}\n"
        f"Energy_Lost_Day_kwh: {daily_line['Energy_Lost_Day_kwh']:.2f}\n"
        f"Energy_Lost_Total_kwh: {daily_line.get('Energy_Lost_Total_kwh', daily_line['Energy_Lost_Day_kwh']):.2f}"
        "</pre>"
    )
    status_visual.value = build_visual_html(daily_line, paused=paused)


async def wait_if_paused(current_day: Optional[dict] = None) -> None:
    while not pause_event.is_set():
        if current_day is not None:
            show_daily_output(current_day, paused=True)
        await pause_event.wait()


async def sleep_with_pause(total_seconds: float, current_day: Optional[dict] = None) -> None:
    elapsed = 0.0
    step = 0.1
    while elapsed < total_seconds:
        await wait_if_paused(current_day)
        remaining = total_seconds - elapsed
        sleep_step = step if remaining > step else remaining
        await asyncio.sleep(sleep_step)
        elapsed += sleep_step


async def run_simulation() -> pd.DataFrame:
    simulation_rows = []
    global battery_storage_kwh
    cumulative_energy_lost_kwh = 0.0

    for row in daily_df.itertuples(index=False):
        await wait_if_paused()

        date = row.date
        wind_kwh_day = row.Windturbine_Produced_kwh_day
        homes_kwh_day = row.homes_shared_energy_usage_kwh_day

        if pd.isna(wind_kwh_day) or pd.isna(homes_kwh_day):
            continue

        wind_kwh_day = float(wind_kwh_day)
        homes_kwh_day = float(homes_kwh_day)

        if wind_kwh_day == 0.0:
            show_daily_output(message="No_Energy_Produced")
            break

        if wind_kwh_day + battery_storage_kwh < homes_kwh_day:
            show_daily_output(message="Energy_Ran_Out")
            break

        battery_before = battery_storage_kwh
        surplus = wind_kwh_day - homes_kwh_day

        if surplus >= 0:
            battery_storage_kwh = min(BATTERY_CAPACITY_KWH, battery_storage_kwh + surplus)
            energy_lost_day_kwh = max(0.0, battery_before + surplus - BATTERY_CAPACITY_KWH)
        else:
            battery_storage_kwh = max(0.0, battery_storage_kwh + surplus)
            energy_lost_day_kwh = 0.0

        cumulative_energy_lost_kwh += energy_lost_day_kwh
        battery_percentage = int(round((battery_storage_kwh / BATTERY_CAPACITY_KWH) * 100))

        daily_line = {
            "date": date,
            "Windturbine_Produced_kwh_day": round(wind_kwh_day, 2),
            "homes_shared_energy_usage_kwh_day": round(homes_kwh_day, 2),
            "Battery_Percentage": battery_percentage,
            "Energy_Lost_Day_kwh": round(energy_lost_day_kwh, 2),
            "Energy_Lost_Total_kwh": round(cumulative_energy_lost_kwh, 2),
        }

        simulation_rows.append(daily_line)
        show_daily_output(daily_line)
        await sleep_with_pause(SECONDS_PER_DAY, daily_line)

    return pd.DataFrame(
        simulation_rows,
        columns=[
            "date",
            "Windturbine_Produced_kwh_day",
            "homes_shared_energy_usage_kwh_day",
            "Battery_Percentage",
            "Energy_Lost_Day_kwh",
            "Energy_Lost_Total_kwh",
        ],
    )


def _on_simulation_done(task: asyncio.Task) -> None:
    try:
        battery_simulation_df = task.result()
        display(HTML(battery_simulation_df.to_html(index=False)))
    except asyncio.CancelledError:
        show_daily_output(message="Simulation cancelled.")
    except Exception as exc:
        show_daily_output(message=f"Simulation failed: {exc}")


existing_task = globals().get("battery_simulation_task")
if isinstance(existing_task, asyncio.Task) and not existing_task.done():
    existing_task.cancel()

battery_simulation_task = asyncio.create_task(run_simulation())
battery_simulation_task.add_done_callback(_on_simulation_done)

ToggleButton(value=False, description='Pause', icon='pause', tooltip='Pause or resume simulation')

date,Windturbine_Produced_kwh_day,homes_shared_energy_usage_kwh_day,Battery_Percentage,Energy_Lost_Day_kwh,Energy_Lost_Total_kwh
2025-01-01,792412.23,77407.61,30,0.00,0.00
2025-01-02,96277.51,43894.65,32,0.00,0.00
2025-01-03,217874.61,85873.11,37,0.00,0.00
2025-01-04,65694.78,69747.62,37,0.00,0.00
2025-01-05,94753.32,9419.20,40,0.00,0.00
2025-01-06,251291.74,97577.37,46,0.00,0.00
2025-01-07,487280.16,76125.78,63,0.00,0.00
2025-01-08,505048.64,78618.62,80,0.00,0.00
2025-01-09,7342.18,12813.35,80,0.00,0.00
2025-01-10,11105.00,45045.58,78,0.00,0.00
